# IMPORT

In [17]:
import sys
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
import numpy as np

# Remonte de deux niveaux (si ton notebook est dans /notebooks/exploration.ipynb)
# pour atteindre la racine du projet
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [18]:
from config.config import *

# Preprocessing

In [19]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


**Les images sont redimensionnées et normalisées selon les statistiques ImageNet, afin d’être compatibles avec le modèle ResNet pré-entraîné**

In [20]:
# Chargement du modèle resnet50 pour création des embeddings.
resnet = models.resnet50(pretrained=True)
# "Supprimer" la couche de classification
resnet.eval()

for param in resnet.parameters():
    param.requires_grad = False

# On créé notre propre modèle 
# *list(resnet.children())[:-1] : récupère toutes les couches de resnet50 sauf la dernière de classification
# nn.Sequential créé un nouveau modèle sur la base des couches et poids récupérés
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])


c:\Users\Fabien\Desktop\OC\P10\BrainScanAI\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Fabien\Desktop\OC\P10\BrainScanAI\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [21]:
def extract_features(image_path, model, transform) -> list :
    img = Image.open(image_path).convert('RGB')
    image = transform(img).unsqueeze(0)
    with torch.no_grad():
        features = model(image)

    return features.squeeze().numpy().tolist()

In [22]:
images_cancer_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "avec_labels" / "cancer"
images_normal_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "avec_labels" / "normal"
images_sans_label_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "sans_label"

In [23]:
features_path = BASE_DIR / "mri_dataset_brain_cancer_oc" / "features.parquet"

# Si le fichier existe déjà, on le charge et on saute le traitement
if features_path.exists():
    print(f"{features_path} exists — chargement depuis le fichier.")
    df_features = pd.read_parquet(features_path)
    features_already_present = True
else:
    features = []

    # On parcourt les dossiers uniquement si le fichier n'existe pas
    for path in tqdm([images_cancer_dir, images_normal_dir, images_sans_label_dir], desc="Progression des dossiers"):
        # On cherche les fichiers .jpg (vos images sont en .jpg)
        for image_path in tqdm(list(path.glob('*.jpg')), desc=f"Traitement {path.name}", leave=False):
            try:
                data = {
                    'name' : image_path.name,
                    'features' : extract_features(image_path, feature_extractor, transform)
                }
                features.append(data)
            except Exception as e:
                print(f"Erreur sur {image_path.name}: {e}")

    # Création du DataFrame
    df_features = pd.DataFrame(features)

    # Afficher la taille d'un embedding si présent
    if not df_features.empty:
        print(f"Forme d'un embedding : {len(df_features['features'].iloc[0])}")

    # S'assurer que la colonne 'features' contient des listes Python
    df_features['features'] = df_features['features'].apply(lambda x: x if isinstance(x, list) else list(x))
    df_features.to_parquet(features_path)
    print(f"Saved embeddings to {features_path}")
    features_already_present = False

Progression des dossiers: 100%|██████████| 3/3 [03:25<00:00, 68.39s/it]


Forme d'un embedding : 2048
Saved embeddings to C:\Users\Fabien\Desktop\OC\P10\BrainScanAI\mri_dataset_brain_cancer_oc\features.parquet
